In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/nafiisaa/cert-dataset/logon.csv
/kaggle/input/datasets/nafiisaa/cert-dataset/file.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Tuesday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Monday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Wednesday-workingHours.pcap_ISCX.csv


In [2]:
# =========================
# STEP 1: DATA ACQUISITION
# =========================

import os
import glob
import gc
import json
import pandas as pd

# -------------------------
# Output setup
# -------------------------
STEP1_OUT_DIR = "/kaggle/working/step1_outputs"
os.makedirs(STEP1_OUT_DIR, exist_ok=True)

MANIFEST_PATH = os.path.join(STEP1_OUT_DIR, "data_manifest.json")
FILE_INDEX_PATH = os.path.join(STEP1_OUT_DIR, "input_file_index.csv")

# -------------------------
# Helpers
# -------------------------
def clean_colname(c):
    c = str(c).strip().lower()
    for ch in [" ", "-", "/", ".", "(", ")", ":"]:
        c = c.replace(ch, "_")
    while "__" in c:
        c = c.replace("__", "_")
    return c.strip("_")

def free_memory(*objs):
    for obj in objs:
        try:
            del obj
        except:
            pass
    gc.collect()

def safe_read_head(path, nrows=3):
    """
    Read a tiny sample only.
    Uses dtype=str to avoid large type inference memory overhead.
    """
    return pd.read_csv(path, nrows=nrows, dtype=str, low_memory=True)

def inspect_csv(path, name="", nrows=3):
    print("=" * 110)
    print("NAME  :", name if name else os.path.basename(path))
    print("PATH  :", path)
    print("EXISTS:", os.path.exists(path))

    if not os.path.exists(path):
        print("Missing file.")
        return {
            "path": path,
            "exists": False,
            "rows_previewed": 0,
            "num_columns": 0,
            "columns_original": [],
            "columns_cleaned": []
        }

    try:
        df = safe_read_head(path, nrows=nrows)
        original_cols = df.columns.tolist()
        cleaned_cols = [clean_colname(c) for c in original_cols]

        print("ROWS PREVIEWED   :", len(df))
        print("NUMBER OF COLUMNS:", len(original_cols))
        print("ORIGINAL COLUMNS :", original_cols)
        print("CLEANED COLUMNS  :", cleaned_cols)

        display(df.head(min(2, len(df))))

        meta = {
            "path": path,
            "exists": True,
            "rows_previewed": int(len(df)),
            "num_columns": int(len(original_cols)),
            "columns_original": original_cols,
            "columns_cleaned": cleaned_cols
        }

        free_memory(df)
        return meta

    except Exception as e:
        print("ERROR READING FILE:", repr(e))
        return {
            "path": path,
            "exists": True,
            "rows_previewed": 0,
            "num_columns": 0,
            "columns_original": [],
            "columns_cleaned": [],
            "error": repr(e)
        }

def find_first_existing_file(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

def find_all_csvs_under(base_dir):
    if not os.path.exists(base_dir):
        return []
    return sorted(glob.glob(os.path.join(base_dir, "**", "*.csv"), recursive=True))

def find_dataset_root_by_keywords(required_files=None, required_keywords=None):
    """
    Search /kaggle/input recursively and return the best matching folder.
    """
    required_files = required_files or []
    required_keywords = [k.lower() for k in (required_keywords or [])]

    candidate_dirs = set()

    for root, dirs, files in os.walk("/kaggle/input"):
        files_lower = {f.lower() for f in files}
        root_lower = root.lower()

        file_match = all(f.lower() in files_lower for f in required_files) if required_files else True
        kw_match = all(k in root_lower for k in required_keywords) if required_keywords else True

        if file_match and kw_match:
            candidate_dirs.add(root)

    if not candidate_dirs:
        return None

    # Prefer the shortest matching path
    return sorted(candidate_dirs, key=lambda x: (len(x), x))[0]

# -------------------------
# 1A. Build input file index
# -------------------------
all_input_files = []
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        full_path = os.path.join(root, f)
        try:
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
        except:
            size_mb = None
        all_input_files.append({
            "path": full_path,
            "filename": f,
            "extension": os.path.splitext(f)[1].lower(),
            "size_mb": round(size_mb, 4) if size_mb is not None else None
        })

file_index_df = pd.DataFrame(all_input_files).sort_values(["path"]).reset_index(drop=True)
file_index_df.to_csv(FILE_INDEX_PATH, index=False)

print("Saved input file index:", FILE_INDEX_PATH)
print("Total files found under /kaggle/input:", len(file_index_df))

# -------------------------
# 1B. Resolve CERT dataset robustly
# -------------------------
cert_root = find_dataset_root_by_keywords(
    required_files=["logon.csv", "file.csv"],
    required_keywords=["cert"]
)

# fallback guesses if keyword-based match fails
if cert_root is None:
    cert_root = find_first_existing_file([
        "/kaggle/input/cert-dataset",
        "/kaggle/input/datasets/nafiisaa/cert-dataset"
    ])
    if cert_root is not None and os.path.isfile(cert_root):
        cert_root = os.path.dirname(cert_root)

CERT_LOGON_PATH = None
CERT_FILE_PATH = None

if cert_root is not None:
    CERT_LOGON_PATH = find_first_existing_file([
        os.path.join(cert_root, "logon.csv"),
        os.path.join(cert_root, "LOGON.csv")
    ])
    CERT_FILE_PATH = find_first_existing_file([
        os.path.join(cert_root, "file.csv"),
        os.path.join(cert_root, "FILE.csv")
    ])

# -------------------------
# 1C. Resolve CIC dataset robustly
# -------------------------
cic_root = find_dataset_root_by_keywords(
    required_keywords=["network", "intrusion"]
)

# fallback guesses
fallback_cic_dirs = [
    "/kaggle/input/network-intrusion-dataset",
    "/kaggle/input/cic-ids-2017",
    "/kaggle/input/datasets/chethuhn/network-intrusion-dataset"
]

if cic_root is None:
    for d in fallback_cic_dirs:
        if os.path.exists(d):
            cic_root = d
            break

CIC_PATHS = find_all_csvs_under(cic_root) if cic_root is not None else []

# Keep likely CIC traffic CSVs only
if CIC_PATHS:
    filtered = []
    for p in CIC_PATHS:
        name = os.path.basename(p).lower()
        if any(x in name for x in [
            "workinghours", "portscan", "ddos", "dos", "bot", "webattack",
            "infilteration", "ftp", "ssh", "heartbleed", "pcap"
        ]):
            filtered.append(p)
    if len(filtered) > 0:
        CIC_PATHS = sorted(filtered)

# -------------------------
# 1D. Show resolved paths
# -------------------------
print("\n" + "=" * 110)
print("RESOLVED DATASET PATHS")
print("=" * 110)
print("CERT ROOT       :", cert_root)
print("CERT_LOGON_PATH :", CERT_LOGON_PATH)
print("CERT_FILE_PATH  :", CERT_FILE_PATH)
print("CIC ROOT        :", cic_root)
print("CIC FILE COUNT  :", len(CIC_PATHS))

for p in CIC_PATHS:
    print(" -", os.path.basename(p))

# -------------------------
# 1E. Inspect files safely
# -------------------------
manifest = {
    "cert_root": cert_root,
    "cic_root": cic_root,
    "cert_logon_path": CERT_LOGON_PATH,
    "cert_file_path": CERT_FILE_PATH,
    "cic_paths": CIC_PATHS,
    "cert_logon_meta": None,
    "cert_file_meta": None,
    "cic_meta": []
}

if CERT_LOGON_PATH:
    manifest["cert_logon_meta"] = inspect_csv(CERT_LOGON_PATH, name="CERT logon.csv", nrows=3)
else:
    print("\nCERT logon.csv not found.")

if CERT_FILE_PATH:
    manifest["cert_file_meta"] = inspect_csv(CERT_FILE_PATH, name="CERT file.csv", nrows=3)
else:
    print("\nCERT file.csv not found.")

if CIC_PATHS:
    print("\nInspecting CIC files safely...")
    for p in CIC_PATHS:
        meta = inspect_csv(p, name=os.path.basename(p), nrows=2)
        manifest["cic_meta"].append(meta)
else:
    print("\nNo CIC CSV files found.")

# -------------------------
# 1F. Hard validation for downstream steps
# -------------------------
issues = []

if CERT_LOGON_PATH is None:
    issues.append("CERT logon.csv not found")
if CERT_FILE_PATH is None:
    issues.append("CERT file.csv not found")
if len(CIC_PATHS) == 0:
    issues.append("No CIC CSV files found")

manifest["issues"] = issues
manifest["status"] = "ok" if len(issues) == 0 else "check_paths"

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", MANIFEST_PATH)

if issues:
    print("\n STEP 1 completed with path issues:")
    for x in issues:
        print(" -", x)
else:
    print("\n STEP 1 COMPLETE: datasets resolved correctly and previewed safely.")

Saved input file index: /kaggle/working/step1_outputs/input_file_index.csv
Total files found under /kaggle/input: 10

RESOLVED DATASET PATHS
CERT ROOT       : /kaggle/input/datasets/nafiisaa/cert-dataset
CERT_LOGON_PATH : /kaggle/input/datasets/nafiisaa/cert-dataset/logon.csv
CERT_FILE_PATH  : /kaggle/input/datasets/nafiisaa/cert-dataset/file.csv
CIC ROOT        : /kaggle/input/datasets/chethuhn/network-intrusion-dataset
CIC FILE COUNT  : 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv
NAME  : CERT logon.csv
PATH  : /kaggle/input/datasets/nafiisaa/cert-dataset/logon.csv
EXISTS: True
ROWS PREVIEWED   : 3
NUMBER OF COLUMNS: 5
ORIGINAL COLUMNS : ['id',

,id,date,user,pc,activity
0,{F3X8-Y2GT43DR-4906OHBL},01/02/2010 02:19:18,DNS1758,PC-0414,Logon
1,{B4Q0-D0GM24KN-3704MAII},01/02/2010 02:31:12,DNS1758,PC-0414,Logoff


NAME  : CERT file.csv
PATH  : /kaggle/input/datasets/nafiisaa/cert-dataset/file.csv
EXISTS: True
ROWS PREVIEWED   : 3
NUMBER OF COLUMNS: 9
ORIGINAL COLUMNS : ['id', 'date', 'user', 'pc', 'filename', 'activity', 'to_removable_media', 'from_removable_media', 'content']
CLEANED COLUMNS  : ['id', 'date', 'user', 'pc', 'filename', 'activity', 'to_removable_media', 'from_removable_media', 'content']


,id,date,user,pc,filename,activity,to_removable_media,from_removable_media,content
0,{F3E2-X3MV05YQ-3516SZDT},01/02/2010 07:19:41,SDH2394,PC-5849,R:\60WBQE7S.doc,File Open,False,True,"D0-CF-11-E0-A1-B1-1A-E1 Ernesztin's brother, L..."
1,{I6N1-Z7VL92UY-8715ESKQ},01/02/2010 07:21:30,SDH2394,PC-5849,R:\0VGILDW8.pdf,File Write,True,False,25-50-44-46-2D ---- Bengali As do many other T...



Inspecting CIC files safely...
NAME  : Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
PATH  : /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
EXISTS: True
ROWS PREVIEWED   : 2
NUMBER OF COLUMNS: 79
ORIGINAL COLUMNS : [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',

...

NAME  : Tuesday-WorkingHours.pcap_ISCX.csv
PATH  : /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Tuesday-WorkingHours.pcap_ISCX.csv
EXISTS: True
ROWS PREVIEWED   : 2
NUMBER OF COLUMNS: 79
ORIGINAL COLUMNS : [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' 

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,88,640,7,4,440,358,220,0,62.85714286,107.349008,...,20,0,0,0,0,0,0,0,0,BENIGN
1,88,900,9,4,600,2944,300,0,66.66666667,132.2875656,...,20,0,0,0,0,0,0,0,0,BENIGN


NAME  : Wednesday-workingHours.pcap_ISCX.csv
PATH  : /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Wednesday-workingHours.pcap_ISCX.csv
EXISTS: True
ROWS PREVIEWED   : 2
NUMBER OF COLUMNS: 79
ORIGINAL COLUMNS : [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s'

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.63636364,31.4492376,...,32,0,0,0,0,0,0,0,0,BENIGN



Saved manifest: /kaggle/working/step1_outputs/data_manifest.json

 STEP 1 COMPLETE: datasets resolved correctly and previewed safely.


In [3]:
# =========================
# STEP 2: DATA CLEANING & NORMALIZATION
# =========================

import os
import gc
import json
import re
import numpy as np
import pandas as pd

# -------------------------
# Paths
# -------------------------
STEP1_OUT_DIR = "/kaggle/working/step1_outputs"
STEP2_OUT_DIR = "/kaggle/working/step2_outputs"
os.makedirs(STEP2_OUT_DIR, exist_ok=True)

MANIFEST_PATH = os.path.join(STEP1_OUT_DIR, "data_manifest.json")
CERT_CLEAN_LOGON_PATH = os.path.join(STEP2_OUT_DIR, "cert_logon_clean.parquet")
CERT_CLEAN_FILE_PATH  = os.path.join(STEP2_OUT_DIR, "cert_file_clean.parquet")
CIC_CLEAN_DIR = os.path.join(STEP2_OUT_DIR, "cic_clean_parts")
os.makedirs(CIC_CLEAN_DIR, exist_ok=True)

STEP2_SUMMARY_PATH = os.path.join(STEP2_OUT_DIR, "step2_summary.json")

# -------------------------
# Load manifest from Step 1
# -------------------------
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

CERT_LOGON_PATH = manifest.get("cert_logon_path")
CERT_FILE_PATH  = manifest.get("cert_file_path")
CIC_PATHS       = manifest.get("cic_paths", [])

print("Loaded Step 1 manifest")
print("CERT_LOGON_PATH:", CERT_LOGON_PATH)
print("CERT_FILE_PATH :", CERT_FILE_PATH)
print("CIC file count :", len(CIC_PATHS))

# -------------------------
# Utilities
# -------------------------
def free_memory(*objs):
    for obj in objs:
        try:
            del obj
        except:
            pass
    gc.collect()

def clean_colname(c):
    c = str(c).strip().lower()
    c = c.replace("(", "_").replace(")", "_")
    c = c.replace("/", "_").replace("\\", "_")
    c = c.replace("-", "_").replace(".", "_")
    c = c.replace(" ", "_").replace(":", "_")
    c = re.sub(r"_+", "_", c).strip("_")
    return c

def standardize_columns(df):
    df = df.copy()
    df.columns = [clean_colname(c) for c in df.columns]
    return df

def normalize_text_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    x = re.sub(r"\s+", "_", x)
    return x

def coerce_numeric_columns(df, exclude=None):
    exclude = set(exclude or [])
    for c in df.columns:
        if c in exclude:
            continue
        if df[c].dtype == object:
            converted = pd.to_numeric(df[c], errors="coerce")
            non_null_ratio = converted.notna().mean()
            # only adopt numeric conversion if it seems meaningful
            if non_null_ratio >= 0.80:
                df[c] = converted
    return df

def safe_datetime_parse(series):
    s1 = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
    return s1

def choose_existing(cols, candidates):
    for cand in candidates:
        if cand in cols:
            return cand
    return None

# -------------------------
# CERT cleaning
# -------------------------
def clean_cert_file(path, out_path, dataset_name):
    if path is None or not os.path.exists(path):
        print(f"Skipping {dataset_name}: file not found")
        return None

    print(f"\nCleaning CERT dataset: {dataset_name}")
    df = pd.read_csv(path, low_memory=True)
    df = standardize_columns(df)

    original_rows, original_cols = df.shape

    # Normalize text columns lightly
    for c in df.columns:
        if df[c].dtype == object:
            # avoid converting very high-cardinality raw fields blindly if huge
            sample_non_null = df[c].dropna().astype(str).head(200)
            avg_len = sample_non_null.str.len().mean() if len(sample_non_null) else 0
            if avg_len < 100:
                df[c] = df[c].map(normalize_text_value)

    # Common CERT aliases
    user_col = choose_existing(df.columns, ["user", "user_id", "employee", "employee_name"])
    pc_col   = choose_existing(df.columns, ["pc", "device", "host", "computer"])
    act_col  = choose_existing(df.columns, ["activity", "action", "op", "event"])
    date_col = choose_existing(df.columns, ["date", "timestamp", "time", "datetime"])

    if date_col is not None:
        df["timestamp"] = safe_datetime_parse(df[date_col])

    # Add source marker
    df["source_dataset"] = "cert"
    df["source_table"] = dataset_name

    # Create unified fields for later steps
    if user_col is not None:
        df["entity_id"] = df[user_col].astype(str)
    else:
        df["entity_id"] = "unknown_user"

    if pc_col is not None:
        df["asset_id"] = df[pc_col].astype(str)
    else:
        df["asset_id"] = "unknown_asset"

    if act_col is not None:
        df["raw_action"] = df[act_col].astype(str)
    else:
        df["raw_action"] = dataset_name

    # Remove exact duplicates
    before_dupes = len(df)
    df = df.drop_duplicates()
    after_dupes = len(df)

    # Save
    df.to_parquet(out_path, index=False)

    summary = {
        "dataset_name": dataset_name,
        "path": path,
        "output_path": out_path,
        "rows_before": int(original_rows),
        "cols_before": int(original_cols),
        "rows_after": int(len(df)),
        "cols_after": int(df.shape[1]),
        "duplicates_removed": int(before_dupes - after_dupes),
        "timestamp_non_null": int(df["timestamp"].notna().sum()) if "timestamp" in df.columns else 0,
        "entity_id_non_null": int(df["entity_id"].notna().sum()),
        "asset_id_non_null": int(df["asset_id"].notna().sum())
    }

    print(summary)
    free_memory(df)
    return summary

# -------------------------
# CIC cleaning
# -------------------------
def map_cic_columns(df):
    """
    Standardize CIC columns to a stable schema without using label-derived features
    for behavior construction.
    """
    cols = set(df.columns)

    mapped = {}

    # common identifiers
    mapped["src_ip_col"] = choose_existing(cols, ["source_ip", "src_ip"])
    mapped["dst_ip_col"] = choose_existing(cols, ["destination_ip", "dst_ip"])
    mapped["src_port_col"] = choose_existing(cols, ["source_port", "src_port"])
    mapped["dst_port_col"] = choose_existing(cols, ["destination_port", "dst_port"])
    mapped["protocol_col"] = choose_existing(cols, ["protocol"])

    # timestamps are inconsistent across CIC files; do NOT invent fake timestamps here
    mapped["timestamp_col"] = choose_existing(cols, ["timestamp", "flow_start_time", "date", "datetime"])

    # label
    mapped["label_col"] = choose_existing(cols, ["label"])

    # core traffic metrics
    mapped["flow_duration_col"] = choose_existing(cols, ["flow_duration"])
    mapped["tot_fwd_pkts_col"] = choose_existing(cols, ["total_fwd_packets", "tot_fwd_pkts"])
    mapped["tot_bwd_pkts_col"] = choose_existing(cols, ["total_backward_packets", "tot_bwd_pkts"])
    mapped["totlen_fwd_col"] = choose_existing(cols, ["total_length_of_fwd_packets", "totlen_fwd_pkts"])
    mapped["totlen_bwd_col"] = choose_existing(cols, ["total_length_of_bwd_packets", "totlen_bwd_pkts"])
    mapped["flow_byts_s_col"] = choose_existing(cols, ["flow_bytes_s"])
    mapped["flow_pkts_s_col"] = choose_existing(cols, ["flow_packets_s"])
    mapped["syn_flag_col"] = choose_existing(cols, ["syn_flag_count"])
    mapped["rst_flag_col"] = choose_existing(cols, ["rst_flag_count"])
    mapped["psh_flag_col"] = choose_existing(cols, ["psh_flag_count"])
    mapped["ack_flag_col"] = choose_existing(cols, ["ack_flag_count"])

    return mapped

def clean_cic_file(path, out_dir):
    print("\nCleaning CIC file:", os.path.basename(path))

    # Read full single file only; do not concatenate all CIC files in memory
    df = pd.read_csv(path, low_memory=True)
    df = standardize_columns(df)

    original_shape = df.shape
    mapping = map_cic_columns(df)

    # Normalize label safely
    label_col = mapping["label_col"]
    if label_col is not None:
        df["label"] = df[label_col].astype(str).map(normalize_text_value)
    else:
        df["label"] = "unknown"

    # Remove rows that are completely empty
    df = df.dropna(how="all")

    # Coerce numerics carefully
    protected_text_cols = [c for c in [
        mapping["src_ip_col"], mapping["dst_ip_col"], mapping["timestamp_col"], "label"
    ] if c is not None]

    df = coerce_numeric_columns(df, exclude=protected_text_cols)

    # Build standardized identifier columns
    if mapping["src_ip_col"] is not None:
        df["src_ip"] = df[mapping["src_ip_col"]].astype(str)
    else:
        df["src_ip"] = "unknown_src_ip"

    if mapping["dst_ip_col"] is not None:
        df["dst_ip"] = df[mapping["dst_ip_col"]].astype(str)
    else:
        df["dst_ip"] = "unknown_dst_ip"

    if mapping["src_port_col"] is not None:
        df["src_port"] = pd.to_numeric(df[mapping["src_port_col"]], errors="coerce")
    else:
        df["src_port"] = np.nan

    if mapping["dst_port_col"] is not None:
        df["dst_port"] = pd.to_numeric(df[mapping["dst_port_col"]], errors="coerce")
    else:
        df["dst_port"] = np.nan

    if mapping["protocol_col"] is not None:
        df["protocol_std"] = df[mapping["protocol_col"]]
    else:
        df["protocol_std"] = np.nan

    # Parse true timestamp only if present; do NOT synthesize time here
    ts_col = mapping["timestamp_col"]
    if ts_col is not None:
        df["timestamp"] = safe_datetime_parse(df[ts_col])
    else:
        df["timestamp"] = pd.NaT

    # Core numeric features for later modeling
    numeric_map = {
        "flow_duration": mapping["flow_duration_col"],
        "total_fwd_packets": mapping["tot_fwd_pkts_col"],
        "total_bwd_packets": mapping["tot_bwd_pkts_col"],
        "total_length_fwd_packets": mapping["totlen_fwd_col"],
        "total_length_bwd_packets": mapping["totlen_bwd_col"],
        "flow_bytes_per_sec": mapping["flow_byts_s_col"],
        "flow_packets_per_sec": mapping["flow_pkts_s_col"],
        "syn_flag_count": mapping["syn_flag_col"],
        "rst_flag_count": mapping["rst_flag_col"],
        "psh_flag_count": mapping["psh_flag_col"],
        "ack_flag_count": mapping["ack_flag_col"],
    }

    for std_col, raw_col in numeric_map.items():
        if raw_col is not None:
            df[std_col] = pd.to_numeric(df[raw_col], errors="coerce")
        else:
            df[std_col] = np.nan

    # Minimal cleanup
    before_dupes = len(df)
    df = df.drop_duplicates()
    after_dupes = len(df)

    # Keep only the columns needed for downstream work
    keep_cols = [
        "src_ip", "dst_ip", "src_port", "dst_port", "protocol_std", "timestamp", "label",
        "flow_duration", "total_fwd_packets", "total_bwd_packets",
        "total_length_fwd_packets", "total_length_bwd_packets",
        "flow_bytes_per_sec", "flow_packets_per_sec",
        "syn_flag_count", "rst_flag_count", "psh_flag_count", "ack_flag_count"
    ]

    slim = df[keep_cols].copy()
    slim["source_dataset"] = "cic_ids2017"
    slim["source_file"] = os.path.basename(path)

    # Important: remove impossible garbage labels formatting
    slim["label"] = slim["label"].fillna("unknown").astype(str).str.strip().str.lower()

    out_name = os.path.splitext(os.path.basename(path))[0] + "_clean.parquet"
    out_path = os.path.join(out_dir, out_name)
    slim.to_parquet(out_path, index=False)

    summary = {
        "input_path": path,
        "output_path": out_path,
        "rows_before": int(original_shape[0]),
        "cols_before": int(original_shape[1]),
        "rows_after": int(len(slim)),
        "cols_after": int(slim.shape[1]),
        "duplicates_removed": int(before_dupes - after_dupes),
        "timestamp_non_null": int(slim["timestamp"].notna().sum()),
        "unique_labels": sorted(slim["label"].dropna().astype(str).unique().tolist())[:50]
    }

    print(summary)
    free_memory(df, slim)
    return summary

# -------------------------
# Run CERT cleaning
# -------------------------
step2_summary = {
    "cert": {},
    "cic": [],
    "notes": [
        "No synthetic timestamps created in Step 2",
        "No behavior abstraction from label created in Step 2",
        "CIC files processed one by one to avoid memory pressure"
    ]
}

cert_logon_summary = clean_cert_file(
    path=CERT_LOGON_PATH,
    out_path=CERT_CLEAN_LOGON_PATH,
    dataset_name="logon"
)
step2_summary["cert"]["logon"] = cert_logon_summary

cert_file_summary = clean_cert_file(
    path=CERT_FILE_PATH,
    out_path=CERT_CLEAN_FILE_PATH,
    dataset_name="file"
)
step2_summary["cert"]["file"] = cert_file_summary

# -------------------------
# Run CIC cleaning file-by-file
# -------------------------
for path in CIC_PATHS:
    try:
        s = clean_cic_file(path, CIC_CLEAN_DIR)
        step2_summary["cic"].append(s)
    except Exception as e:
        err = {
            "input_path": path,
            "error": repr(e)
        }
        print("ERROR:", err)
        step2_summary["cic"].append(err)
    gc.collect()

# -------------------------
# Save summary
# -------------------------
with open(STEP2_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(step2_summary, f, indent=2)

print("\nSaved Step 2 summary:", STEP2_SUMMARY_PATH)

print("\n STEP 2 COMPLETE")
print("Outputs:")
print(" -", CERT_CLEAN_LOGON_PATH)
print(" -", CERT_CLEAN_FILE_PATH)
print(" -", CIC_CLEAN_DIR)

Loaded Step 1 manifest
CERT_LOGON_PATH: /kaggle/input/datasets/nafiisaa/cert-dataset/logon.csv
CERT_FILE_PATH : /kaggle/input/datasets/nafiisaa/cert-dataset/file.csv
CIC file count : 8

Cleaning CERT dataset: logon


/tmp/ipykernel_23/2359958230.py:88: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  s1 = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_23/2359958230.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s1 = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)


{'dataset_name': 'logon', 'path': '/kaggle/input/datasets/nafiisaa/cert-dataset/logon.csv', 'output_path': '/kaggle/working/step2_outputs/cert_logon_clean.parquet', 'rows_before': 3530285, 'cols_before': 5, 'rows_after': 3530285, 'cols_after': 11, 'duplicates_removed': 0, 'timestamp_non_null': 0, 'entity_id_non_null': 3530285, 'asset_id_non_null': 3530285}

Cleaning CERT dataset: file


/tmp/ipykernel_23/2359958230.py:88: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  s1 = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_23/2359958230.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s1 = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)


{'dataset_name': 'file', 'path': '/kaggle/input/datasets/nafiisaa/cert-dataset/file.csv', 'output_path': '/kaggle/working/step2_outputs/cert_file_clean.parquet', 'rows_before': 2014883, 'cols_before': 9, 'rows_after': 2014883, 'cols_after': 15, 'duplicates_removed': 0, 'timestamp_non_null': 0, 'entity_id_non_null': 2014883, 'asset_id_non_null': 2014883}

Cleaning CIC file: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
{'input_path': '/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'output_path': '/kaggle/working/step2_outputs/cic_clean_parts/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_clean.parquet', 'rows_before': 225745, 'cols_before': 79, 'rows_after': 223112, 'cols_after': 20, 'duplicates_removed': 2633, 'timestamp_non_null': 0, 'unique_labels': ['benign', 'ddos']}

Cleaning CIC file: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
{'input_path': '/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-W

In [4]:
# =========================
# STEP 3: BEHAVIORAL ABSTRACTION LAYER
# =========================

import os
import gc
import json
import numpy as np
import pandas as pd

# -------------------------
# Paths
# -------------------------
STEP2_OUT_DIR = "/kaggle/working/step2_outputs"
STEP3_OUT_DIR = "/kaggle/working/step3_outputs"
os.makedirs(STEP3_OUT_DIR, exist_ok=True)

CERT_LOGON_CLEAN = os.path.join(STEP2_OUT_DIR, "cert_logon_clean.parquet")
CERT_FILE_CLEAN  = os.path.join(STEP2_OUT_DIR, "cert_file_clean.parquet")
CIC_CLEAN_DIR    = os.path.join(STEP2_OUT_DIR, "cic_clean_parts")

CERT_LOGON_ABS = os.path.join(STEP3_OUT_DIR, "cert_logon_behavior.parquet")
CERT_FILE_ABS  = os.path.join(STEP3_OUT_DIR, "cert_file_behavior.parquet")
CIC_ABS_DIR    = os.path.join(STEP3_OUT_DIR, "cic_behavior_parts")
os.makedirs(CIC_ABS_DIR, exist_ok=True)

STEP3_SUMMARY_PATH = os.path.join(STEP3_OUT_DIR, "step3_summary.json")

# -------------------------
# Utilities
# -------------------------
def free_memory(*objs):
    for obj in objs:
        try:
            del obj
        except:
            pass
    gc.collect()

def safe_str(x, default="unknown"):
    if pd.isna(x):
        return default
    s = str(x).strip().lower()
    return s if s else default

def safe_num(series, fill=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(fill)

# -------------------------
# CERT behavior abstraction
# -------------------------
def abstract_cert_logon(df):
    df = df.copy()

    # Standardized raw action text
    raw_action = df["raw_action"].astype(str).str.lower().fillna("unknown")

    def map_logon_behavior(x):
        x = str(x).lower()
        if "logon" in x or "login" in x:
            return "authentication_logon"
        if "logoff" in x or "logout" in x:
            return "authentication_logoff"
        if "connect" in x or "remote" in x or "vpn" in x:
            return "remote_access"
        if "fail" in x or "denied" in x:
            return "authentication_failure"
        return "authentication_other"

    df["behavior_type"] = raw_action.map(map_logon_behavior)
    df["behavior_family"] = "authentication"

    # Entity/session anchors
    df["entity_id"] = df["entity_id"].fillna("unknown_user").astype(str)
    df["asset_id"] = df["asset_id"].fillna("unknown_asset").astype(str)

    keep = [
        "timestamp", "entity_id", "asset_id", "raw_action",
        "behavior_type", "behavior_family", "source_dataset", "source_table"
    ]
    keep = [c for c in keep if c in df.columns]
    return df[keep].copy()

def abstract_cert_file(df):
    df = df.copy()

    raw_action = df["raw_action"].astype(str).str.lower().fillna("unknown")

    def map_file_behavior(x):
        x = str(x).lower()
        if "copy" in x:
            return "file_copy"
        if "write" in x or "save" in x or "create" in x:
            return "file_write"
        if "read" in x or "open" in x:
            return "file_read"
        if "delete" in x or "remove" in x:
            return "file_delete"
        if "rename" in x or "move" in x:
            return "file_move"
        return "file_other"

    df["behavior_type"] = raw_action.map(map_file_behavior)
    df["behavior_family"] = "file_activity"

    df["entity_id"] = df["entity_id"].fillna("unknown_user").astype(str)
    df["asset_id"] = df["asset_id"].fillna("unknown_asset").astype(str)

    keep = [
        "timestamp", "entity_id", "asset_id", "raw_action",
        "behavior_type", "behavior_family", "source_dataset", "source_table"
    ]
    keep = [c for c in keep if c in df.columns]
    return df[keep].copy()

def process_cert_behavior(input_path, output_path, mode):
    if not os.path.exists(input_path):
        print(f"Missing CERT file: {input_path}")
        return {"input_path": input_path, "missing": True}

    df = pd.read_parquet(input_path)

    if mode == "logon":
        out = abstract_cert_logon(df)
    elif mode == "file":
        out = abstract_cert_file(df)
    else:
        raise ValueError(f"Unknown CERT mode: {mode}")

    out.to_parquet(output_path, index=False)

    summary = {
        "input_path": input_path,
        "output_path": output_path,
        "rows": int(len(out)),
        "columns": int(out.shape[1]),
        "behavior_counts": out["behavior_type"].value_counts(dropna=False).to_dict()
    }

    print(f"\nCERT {mode} behavior summary:")
    print(summary)

    free_memory(df, out)
    return summary

# -------------------------
# CIC behavior abstraction
# -------------------------
def abstract_cic_behavior(df):
    """
    IMPORTANT:
    - No mapping from label -> behavior
    - Behavior inferred only from flow statistics
    - Label preserved only for evaluation in later steps
    """
    df = df.copy()

    # Safe numeric fields
    df["flow_duration"] = safe_num(df.get("flow_duration", 0))
    df["total_fwd_packets"] = safe_num(df.get("total_fwd_packets", 0))
    df["total_bwd_packets"] = safe_num(df.get("total_bwd_packets", 0))
    df["total_length_fwd_packets"] = safe_num(df.get("total_length_fwd_packets", 0))
    df["total_length_bwd_packets"] = safe_num(df.get("total_length_bwd_packets", 0))
    df["flow_bytes_per_sec"] = safe_num(df.get("flow_bytes_per_sec", 0))
    df["flow_packets_per_sec"] = safe_num(df.get("flow_packets_per_sec", 0))
    df["syn_flag_count"] = safe_num(df.get("syn_flag_count", 0))
    df["rst_flag_count"] = safe_num(df.get("rst_flag_count", 0))
    df["psh_flag_count"] = safe_num(df.get("psh_flag_count", 0))
    df["ack_flag_count"] = safe_num(df.get("ack_flag_count", 0))

    # Safe identifiers
    df["src_ip"] = df.get("src_ip", "unknown_src").astype(str)
    df["dst_ip"] = df.get("dst_ip", "unknown_dst").astype(str)
    df["protocol_std"] = df.get("protocol_std", "unknown").astype(str)
    df["label"] = df.get("label", "unknown").astype(str)

    # Derived statistics
    df["total_packets"] = df["total_fwd_packets"] + df["total_bwd_packets"]
    df["total_bytes"] = df["total_length_fwd_packets"] + df["total_length_bwd_packets"]
    df["fwd_ratio"] = np.where(
        df["total_packets"] > 0,
        df["total_fwd_packets"] / df["total_packets"],
        0.0
    )

    df["bytes_per_packet"] = np.where(
        df["total_packets"] > 0,
        df["total_bytes"] / df["total_packets"],
        0.0
    )

    # Stateless behavior abstraction from traffic patterns only
    def infer_network_behavior(row):
        pkt_rate = row["flow_packets_per_sec"]
        byte_rate = row["flow_bytes_per_sec"]
        syn_cnt = row["syn_flag_count"]
        rst_cnt = row["rst_flag_count"]
        psh_cnt = row["psh_flag_count"]
        ack_cnt = row["ack_flag_count"]
        total_pkts = row["total_packets"]
        total_bytes = row["total_bytes"]
        fwd_ratio = row["fwd_ratio"]
        bpp = row["bytes_per_packet"]
        dst_port = row["dst_port"] if "dst_port" in row.index else np.nan

        # scan-like: many small packets, high forward dominance, often low bytes/pkt
        if total_pkts >= 10 and fwd_ratio >= 0.8 and bpp <= 120:
            return "scan_like"

        # syn flood / handshake abuse pattern
        if syn_cnt >= 2 and ack_cnt <= syn_cnt and total_pkts >= 3:
            return "syn_like"

        # reset-heavy abnormal connection pattern
        if rst_cnt >= 1 and total_pkts <= 10:
            return "reset_anomaly"

        # bulk transfer pattern
        if total_bytes >= 100000 or byte_rate >= 50000:
            return "bulk_transfer"

        # interactive web/app traffic pattern
        if psh_cnt >= 1 and ack_cnt >= 1 and 40 <= bpp <= 1500:
            return "interactive"

        # dns-ish small exchange
        if dst_port == 53 and total_pkts <= 6:
            return "dns_like"

        # short low-volume connection
        if total_pkts <= 3 and total_bytes <= 300:
            return "short_connection"

        return "general_flow"

    df["behavior_type"] = df.apply(infer_network_behavior, axis=1)

    def behavior_family_map(x):
        if x in ["scan_like", "syn_like", "reset_anomaly"]:
            return "suspicious_network"
        if x in ["bulk_transfer"]:
            return "transfer"
        if x in ["interactive", "dns_like", "short_connection", "general_flow"]:
            return "network_activity"
        return "other"

    df["behavior_family"] = df["behavior_type"].map(behavior_family_map)

    # A stable identity key without forcing it to be a session yet
    df["flow_key"] = (
        df["src_ip"].astype(str) + "|" +
        df["dst_ip"].astype(str) + "|" +
        df["protocol_std"].astype(str)
    )

    keep = [
        "timestamp",
        "src_ip", "dst_ip", "src_port", "dst_port", "protocol_std",
        "flow_key",
        "flow_duration", "total_fwd_packets", "total_bwd_packets",
        "total_length_fwd_packets", "total_length_bwd_packets",
        "flow_bytes_per_sec", "flow_packets_per_sec",
        "syn_flag_count", "rst_flag_count", "psh_flag_count", "ack_flag_count",
        "total_packets", "total_bytes", "fwd_ratio", "bytes_per_packet",
        "behavior_type", "behavior_family",
        "label", "source_dataset", "source_file"
    ]
    keep = [c for c in keep if c in df.columns]
    return df[keep].copy()

def process_cic_behavior(input_path, output_dir):
    df = pd.read_parquet(input_path)
    out = abstract_cic_behavior(df)

    out_path = os.path.join(output_dir, os.path.basename(input_path).replace("_clean.parquet", "_behavior.parquet"))
    out.to_parquet(out_path, index=False)

    summary = {
        "input_path": input_path,
        "output_path": out_path,
        "rows": int(len(out)),
        "columns": int(out.shape[1]),
        "behavior_counts": out["behavior_type"].value_counts(dropna=False).head(20).to_dict(),
        "label_counts": out["label"].value_counts(dropna=False).head(20).to_dict(),
        "timestamp_non_null": int(out["timestamp"].notna().sum()) if "timestamp" in out.columns else 0
    }

    print("\nCIC behavior summary:")
    print(summary)

    free_memory(df, out)
    return summary

# -------------------------
# Run Step 3
# -------------------------
step3_summary = {
    "cert_logon": None,
    "cert_file": None,
    "cic_parts": [],
    "notes": [
        "Behavior is inferred from actions/statistics, not from attack labels",
        "CIC timestamps remain unused for temporal logic when null",
        "Flow key is identity-only, not a true session yet"
    ]
}

# CERT
step3_summary["cert_logon"] = process_cert_behavior(CERT_LOGON_CLEAN, CERT_LOGON_ABS, mode="logon")
step3_summary["cert_file"]  = process_cert_behavior(CERT_FILE_CLEAN, CERT_FILE_ABS, mode="file")

# CIC
cic_inputs = sorted([
    os.path.join(CIC_CLEAN_DIR, f)
    for f in os.listdir(CIC_CLEAN_DIR)
    if f.endswith("_clean.parquet")
])

for p in cic_inputs:
    try:
        s = process_cic_behavior(p, CIC_ABS_DIR)
        step3_summary["cic_parts"].append(s)
    except Exception as e:
        err = {"input_path": p, "error": repr(e)}
        print("ERROR:", err)
        step3_summary["cic_parts"].append(err)
    gc.collect()

# Save summary
with open(STEP3_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(step3_summary, f, indent=2)

print("\nSaved Step 3 summary:", STEP3_SUMMARY_PATH)
print("\n STEP 3 COMPLETE")
print("Outputs:")
print(" -", CERT_LOGON_ABS)
print(" -", CERT_FILE_ABS)
print(" -", CIC_ABS_DIR)


CERT logon behavior summary:
{'input_path': '/kaggle/working/step2_outputs/cert_logon_clean.parquet', 'output_path': '/kaggle/working/step3_outputs/cert_logon_behavior.parquet', 'rows': 3530285, 'columns': 8, 'behavior_counts': {'authentication_logon': 1948933, 'authentication_logoff': 1581352}}

CERT file behavior summary:
{'input_path': '/kaggle/working/step2_outputs/cert_file_clean.parquet', 'output_path': '/kaggle/working/step3_outputs/cert_file_behavior.parquet', 'rows': 2014883, 'columns': 8, 'behavior_counts': {'file_copy': 725458, 'file_read': 657500, 'file_delete': 380799, 'file_write': 251126}}

CIC behavior summary:
{'input_path': '/kaggle/working/step2_outputs/cic_clean_parts/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_clean.parquet', 'output_path': '/kaggle/working/step3_outputs/cic_behavior_parts/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_behavior.parquet', 'rows': 223112, 'columns': 27, 'behavior_counts': {'general_flow': 153631, 'bulk_transfer': 44991, 'dns_like': 1

In [5]:
# =========================
# STEP 4: TEMPORAL / SESSION CONSTRUCTION 
# =========================

import os
import gc
import json
import pandas as pd

# -------------------------
# Paths
# -------------------------
STEP3_OUT_DIR = "/kaggle/working/step3_outputs"
STEP4_OUT_DIR = "/kaggle/working/step4_outputs"
os.makedirs(STEP4_OUT_DIR, exist_ok=True)

CERT_LOGON_PATH = os.path.join(STEP3_OUT_DIR, "cert_logon_behavior.parquet")
CERT_FILE_PATH  = os.path.join(STEP3_OUT_DIR, "cert_file_behavior.parquet")
CIC_DIR         = os.path.join(STEP3_OUT_DIR, "cic_behavior_parts")

CERT_SESSION_OUT = os.path.join(STEP4_OUT_DIR, "cert_sessions.parquet")
CIC_SESSION_DIR  = os.path.join(STEP4_OUT_DIR, "cic_sessions")
os.makedirs(CIC_SESSION_DIR, exist_ok=True)

SUMMARY_PATH = os.path.join(STEP4_OUT_DIR, "step4_summary.json")

# -------------------------
# UTIL
# -------------------------
def free_memory(*objs):
    for obj in objs:
        try:
            del obj
        except:
            pass
    gc.collect()

# =========================
# CERT SESSIONIZATION (REAL TEMPORAL)
# =========================
def build_cert_sessions():

    print("\n[INFO] Building CERT sessions (FINAL FIX)...")

    logon = pd.read_parquet(CERT_LOGON_PATH)
    file  = pd.read_parquet(CERT_FILE_PATH)

    df = pd.concat([logon, file], ignore_index=True)

    # Fallback ordering per user
    df["order"] = df.groupby("entity_id").cumcount()

    df = df.sort_values(["entity_id", "order"])

    WINDOW_SIZE = 30   # realistic human activity window
    STRIDE = 15

    sessions = []

    for user, group in df.groupby("entity_id"):

        behaviors = group["behavior_type"].tolist()

        for i in range(0, len(behaviors) - WINDOW_SIZE, STRIDE):
            seq = behaviors[i:i + WINDOW_SIZE]

            if len(seq) == WINDOW_SIZE:
                sessions.append(seq)

    session_df = pd.DataFrame({
        "session_id": range(len(sessions)),
        "sequence": sessions,
        "length": WINDOW_SIZE,
        "source": "cert"
    })

    session_df.to_parquet(CERT_SESSION_OUT, index=False)

    summary = {
        "num_sessions": len(session_df),
        "avg_length": WINDOW_SIZE
    }

    print("[CERT FINAL SUMMARY]:", summary)

    free_memory(logon, file, df, session_df)

    return summary
    
# =========================
# CIC SESSIONIZATION (FIXED — SLIDING WINDOWS)
# =========================
def build_cic_sessions():

    print("\n[INFO] Building CIC sessions (FIXED)...")

    WINDOW_SIZE = 20
    STRIDE = 10

    summaries = []

    for file in os.listdir(CIC_DIR):
        if not file.endswith(".parquet"):
            continue

        path = os.path.join(CIC_DIR, file)
        df = pd.read_parquet(path)

        # Shuffle to avoid ordering bias
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)

        behaviors = df["behavior_type"].tolist()

        sequences = []

        for i in range(0, len(behaviors) - WINDOW_SIZE, STRIDE):
            seq = behaviors[i:i + WINDOW_SIZE]

            if len(seq) == WINDOW_SIZE:
                sequences.append(seq)

        out_df = pd.DataFrame({
            "session_id": range(len(sequences)),
            "sequence": sequences,
            "length": WINDOW_SIZE,
            "source": "cic",
            "file": file
        })

        out_path = os.path.join(
            CIC_SESSION_DIR,
            file.replace(".parquet", "_sessions.parquet")
        )
        out_df.to_parquet(out_path, index=False)

        summary = {
            "file": file,
            "num_sessions": len(out_df),
            "avg_length": WINDOW_SIZE
        }

        print("[CIC SUMMARY]:", summary)

        summaries.append(summary)

        free_memory(df, out_df)

    return summaries


# =========================
# RUN STEP 4
# =========================
summary = {}

summary["cert"] = build_cert_sessions()
summary["cic"] = build_cic_sessions()

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:", SUMMARY_PATH)
print("\n STEP 4 COMPLETE (FIXED)")


[INFO] Building CERT sessions (FINAL FIX)...
[CERT FINAL SUMMARY]: {'num_sessions': 363593, 'avg_length': 30}

[INFO] Building CIC sessions (FIXED)...
[CIC SUMMARY]: {'file': 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_behavior.parquet', 'num_sessions': 16428, 'avg_length': 20}
[CIC SUMMARY]: {'file': 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_behavior.parquet', 'num_sessions': 25296, 'avg_length': 20}
[CIC SUMMARY]: {'file': 'Monday-WorkingHours.pcap_ISCX_behavior.parquet', 'num_sessions': 50297, 'avg_length': 20}
[CIC SUMMARY]: {'file': 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_behavior.parquet', 'num_sessions': 22310, 'avg_length': 20}
[CIC SUMMARY]: {'file': 'Wednesday-workingHours.pcap_ISCX_behavior.parquet', 'num_sessions': 61078, 'avg_length': 20}
[CIC SUMMARY]: {'file': 'Friday-WorkingHours-Morning.pcap_ISCX_behavior.parquet', 'num_sessions': 18413, 'avg_length': 20}
[CIC SUMMARY]: {'file': 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_behavior.pa